# nano-prover: train on Kaggle (free T4), verify locally

**Before running — in the right-hand panel:**
1. **Settings → Accelerator → `GPU T4 x2`**
2. **Settings → Internet → On**  (needed to clone + pip install + download the model)

Then run both cells. The **only thing you must edit** is the `TOKEN` in cell 2.
Kaggle gives ~30 GPU-hours/week, so you won't hit Colab's 'insufficient quota' wall.

In [ ]:
# 1. GPU check + clone + install + train (0.5B, ~10 min) + generate proofs — all in one.
!nvidia-smi -L
import os
os.chdir('/kaggle/working')
!rm -rf Conjecture-Gym
!git clone --depth 1 https://github.com/FELIXGAO0315/Conjecture-Gym.git
os.chdir('/kaggle/working/Conjecture-Gym')
!pip install -q -U transformers trl peft datasets accelerate
!python training/prepare_data.py
!SFT_MODEL=Qwen/Qwen2.5-Coder-0.5B-Instruct python training/sft_train.py
!python training/gen_preds.py --n 100 --k 1

In [ ]:
# 2. push the small preds.jsonl back to GitHub.  EDIT THE TOKEN (keep the quotes).
#    Make one: github.com -> Settings -> Developer settings -> Personal access tokens
#              -> Tokens (classic) -> Generate new token -> scope: repo -> copy the ghp_... string.
TOKEN = 'ghp_PUT_YOUR_REAL_TOKEN_HERE'
import os
os.chdir('/kaggle/working/Conjecture-Gym')
!git add -f training/preds.jsonl
!git -c user.email=you@example.com -c user.name=you commit -m 'eval preds'
!git push https://{TOKEN}@github.com/FELIXGAO0315/Conjecture-Gym.git main

Then on your **local** machine (where the Lean verifier lives):
```bash
git pull
python training/verify_preds.py     # real Lean-verified pass@k
```